In [11]:
import os
os.chdir(r"C:\vscode\graph-rag\Source")

In [12]:
from config.settings_loader import load_config

config = load_config("config/config.yaml")

In [4]:
import pdfplumber

text = []
with pdfplumber.open(config["data_source"]["raw_data"]["pdf_path"]) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            text.append(page_text)

raw_text = "\n\n".join(text)

In [ ]:
with open(config["data_source"]["normalized_data"]["unstructured_text_path"], "w", encoding="utf-8") as f:
    f.write(raw_text)

In [15]:
import re

INPUT_FILE = "gibbon_raw.md"
OUTPUT_FILE = "gibbon_clean.md"

def is_heading(line: str) -> bool:
    return bool(re.match(r"^Chapter\s+[IVXLC]+:", line)) or \
           bool(re.match(r"^Introduction\.$", line)) or \
           bool(re.match(r"^.*—Part\s+[IVXLC]+\.$", line))

def is_footnote_start(line: str) -> bool:
    return bool(re.match(r"^\d+[a-z]?$", line.strip())) or \
           "(return)" in line

def normalize(text: str) -> str:
    lines = text.splitlines()

    cleaned_lines = []
    buffer = []
    skipping_footnote = False

    for line in lines:
        line = line.rstrip()

        # ---- Skip footnotes completely ----
        if skipping_footnote:
            if line.strip() == "":
                skipping_footnote = False
            continue

        if is_footnote_start(line):
            skipping_footnote = True
            continue

        if line.strip().startswith("[") and line.strip().endswith("]"):
            continue

        # ---- Preserve headings as standalone lines ----
        if is_heading(line):
            if buffer:
                cleaned_lines.append(" ".join(buffer))
                buffer = []
            cleaned_lines.append(line)
            cleaned_lines.append("")  # paragraph break
            continue

        # ---- Paragraph handling ----
        if line.strip() == "":
            if buffer:
                cleaned_lines.append(" ".join(buffer))
                buffer = []
            cleaned_lines.append("")
        else:
            buffer.append(line)

    if buffer:
        cleaned_lines.append(" ".join(buffer))

    return "\n".join(cleaned_lines)

In [ ]:
with open(config["data_source"]["normalized_data"]["unstructured_text_path"], "r", encoding="utf-8") as f:
        raw_text = f.read()

        cleaned_text = normalize(raw_text)

with open(config["data_source"]["normalized_data"]["structured_text_path"], "w", encoding="utf-8") as f:
    f.write(cleaned_text)

print("✅ Normalization complete. Output written to gibbon_clean.md")